In [ ]:
# Сохранение V2 данных и моделей
v2_data_dir = ROOT / 'saved_data' / 'v2'
os.makedirs(v2_data_dir, exist_ok=True)
df_v2.to_csv(str(v2_data_dir / 'combined_v2.csv'), index=False)
print(f'V2 data saved to {v2_data_dir / "combined_v2.csv"}: {len(df_v2)} rows, {len(df_v2.columns)} cols')

v2_model_dir = ROOT / 'saved_models' / 'v2'
os.makedirs(v2_model_dir, exist_ok=True)

# Save XGBoost models
import joblib
for target, res in v2_results.items():
    if 'model' in res:
        path = v2_model_dir / f'xgb_{target}.json'
        res['model'].save_model(str(path))
        print(f'  Saved {path.name}')

# Save DNN
torch.save({
    'model_state': best_state,
    'scaler_mean': scaler_v2.mean_,
    'scaler_scale': scaler_v2.scale_,
    'target_scaler_mean': target_scaler.mean_,
    'target_scaler_scale': target_scaler.scale_,
    'feature_cols': feature_cols_v2,
    'reg_targets': reg_targets,
    'hidden_dims': (256, 128, 64),
}, str(v2_model_dir / 'dnn_multioutput.pt'))
print(f'  Saved dnn_multioutput.pt')

# Final summary
print('\n' + '=' * 70)
print('V2 MULTI-PROCESS LENR TRAINING SUMMARY')
print('=' * 70)
print(f'\nDataset: {len(df_v2)} rows × {len(df_v2.columns)} columns')
print(f'Features: {len(feature_cols_v2)} (8 groups)')
print(f'Classification targets: {len(cls_targets)}')
print(f'Regression targets: {len(reg_targets)}')
print(f'\nData sources:')
for src, cnt in df_v2['data_source'].value_counts().items():
    print(f'  {src}: {cnt}')
print(f'\nXGBoost Multi-Target:')
for target, res in v2_results.items():
    auc_str = f'{res["auc"]:.3f}' if res['auc'] is not None else 'N/A'
    print(f'  {target:20s}: AUC={auc_str}, positive={res["pos_rate"]:.2%}')
print(f'\nDNN Multi-Output: best val loss={best_val:.4f}')
for i, t in enumerate(reg_targets):
    r2 = r2_score(Y_test[:, i], Y_pred[:, i])
    print(f'  {t}: R²={r2:.3f}')
print(f'\nPhysics modes: maxwell, coulomb_original, cherepanov')
print(f'Materials: {len(df_v2["material"].unique())}')

### 10.4 V2 Итоговая сводка

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class MultiOutputDNN(nn.Module):
    """Multi-output DNN for V2 regression targets."""
    def __init__(self, n_features, hidden_dims=(256, 128, 64), n_outputs=4):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)])
            in_dim = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_outputs)

    def forward(self, x):
        return self.head(self.backbone(x))

# Prepare data
Y_reg = df_v2[reg_targets].fillna(0).values.astype(np.float32)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_v2_scaled, Y_reg, test_size=0.2, random_state=42
)

# Normalize targets
from sklearn.preprocessing import StandardScaler as SS
target_scaler = SS()
Y_train_s = target_scaler.fit_transform(Y_train)
Y_test_s = target_scaler.transform(Y_test)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

X_tr = torch.tensor(X_train, dtype=torch.float32)
Y_tr = torch.tensor(Y_train_s, dtype=torch.float32)
X_te = torch.tensor(X_test, dtype=torch.float32)
Y_te = torch.tensor(Y_test_s, dtype=torch.float32)

train_ds = TensorDataset(X_tr, Y_tr)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

model_dnn = MultiOutputDNN(len(feature_cols_v2), hidden_dims=(256, 128, 64), n_outputs=len(reg_targets)).to(device)
optimizer = torch.optim.AdamW(model_dnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion = nn.MSELoss()

# Training loop
train_losses, val_losses = [], []
best_val = float('inf')
patience, patience_counter = 30, 0

for epoch in range(300):
    model_dnn.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model_dnn(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_losses.append(epoch_loss / len(X_tr))

    model_dnn.eval()
    with torch.no_grad():
        val_pred = model_dnn(X_te.to(device))
        val_loss = criterion(val_pred, Y_te.to(device)).item()
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model_dnn.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            break

model_dnn.load_state_dict(best_state)
print(f'Training done: {len(train_losses)} epochs, best val loss: {best_val:.4f}')

# Evaluate
model_dnn.eval()
with torch.no_grad():
    Y_pred_s = model_dnn(X_te.to(device)).cpu().numpy()
Y_pred = target_scaler.inverse_transform(Y_pred_s)

from sklearn.metrics import r2_score, mean_absolute_error

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, target in enumerate(reg_targets):
    r2 = r2_score(Y_test[:, i], Y_pred[:, i])
    mae = mean_absolute_error(Y_test[:, i], Y_pred[:, i])
    axes[i].scatter(Y_test[:, i], Y_pred[:, i], alpha=0.3, s=5, color='#4fc3f7')
    max_val = max(Y_test[:, i].max(), Y_pred[:, i].max())
    axes[i].plot([0, max_val], [0, max_val], 'r--', alpha=0.7)
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')
    axes[i].set_title(f'{target}\nR²={r2:.3f}, MAE={mae:.3f}')
plt.suptitle('V2 Multi-Output DNN — Pred vs Actual', fontsize=14)
plt.tight_layout()
plt.show()

# Training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train', alpha=0.8)
ax.plot(val_losses, label='Validation', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('V2 Multi-Output DNN Training')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

### 10.3 DNN Multi-Output Regressor (V2)

Обучаем DNN с multi-output head для предсказания:
- `excess_heat_W`, `COP`, `neutron_rate_ns`, `energy_density_MJ_kg`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

# Подготовка V2 данных
X_v2 = df_v2[feature_cols_v2].fillna(0).values.astype(np.float32)

scaler_v2 = StandardScaler()
X_v2_scaled = scaler_v2.fit_transform(X_v2)

# Multi-target XGBoost
v2_results = {}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('V2 Multi-Target XGBoost — Feature Importance (top 15)', fontsize=14)

for idx, target in enumerate(cls_targets):
    if target not in df_v2.columns:
        continue

    y = df_v2[target].values.astype(int)
    pos_rate = y.mean()

    # Skip if too imbalanced (<1% positive)
    if pos_rate < 0.005:
        print(f'{target}: skipped (positive rate {pos_rate:.3%})')
        v2_results[target] = {'auc': None, 'f1': None, 'pos_rate': pos_rate}
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_v2_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scale positive weight for imbalanced targets
    scale_pos = max((1 - pos_rate) / pos_rate, 1.0)

    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=min(scale_pos, 20),
        eval_metric='logloss', random_state=42,
        use_label_encoder=False,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = 0.5

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    f1 = report.get('1', report.get('1.0', {})).get('f1-score', 0)

    v2_results[target] = {'auc': auc, 'f1': f1, 'pos_rate': pos_rate, 'model': model}
    print(f'{target}: AUC={auc:.3f}, F1={f1:.3f}, pos_rate={pos_rate:.3%}')

    # Feature importance plot
    row, col = idx // 3, idx % 3
    imp = dict(zip(feature_cols_v2, model.feature_importances_))
    imp_sorted = dict(sorted(imp.items(), key=lambda x: x[1], reverse=True)[:15])
    axes[row, col].barh(range(len(imp_sorted)), list(imp_sorted.values()), color='#4fc3f7')
    axes[row, col].set_yticks(range(len(imp_sorted)))
    axes[row, col].set_yticklabels([k[:20] for k in imp_sorted.keys()], fontsize=7)
    axes[row, col].set_title(f'{target} (AUC={auc:.3f})')
    axes[row, col].invert_yaxis()

plt.tight_layout()
plt.show()

# Summary table
print('\n=== V2 Multi-Target Summary ===')
for target, res in v2_results.items():
    auc_str = f'{res["auc"]:.3f}' if res['auc'] is not None else 'N/A'
    f1_str = f'{res["f1"]:.3f}' if res['f1'] is not None else 'N/A'
    print(f'  {target:20s}: AUC={auc_str}, F1={f1_str}, positive={res["pos_rate"]:.2%}')

### 10.2 XGBoost Multi-Target Classifier (V2)

Обучаем отдельный XGBoost на каждый из 6 classification targets:
- `has_reaction`, `has_excess_heat`, `has_neutrons`, `has_tritium`, `has_helium4`, `has_transmutation`

In [ ]:
# V2: Визуализация многопроцессных фич
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('V2 Multi-Process Feature Distributions', fontsize=16)

# 1. Target distribution — multi-class
target_counts = {t: df_v2[t].sum() for t in cls_targets if t in df_v2.columns}
axes[0,0].bar(target_counts.keys(), target_counts.values(), color='#4fc3f7')
axes[0,0].set_title('Classification Targets (positive counts)')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Screening energy vs barrier reduction (3 modes)
for mode, color in [('maxwell', '#4fc3f7'), ('coulomb', '#ff7043'), ('cherepanov', '#66bb6a')]:
    col = f'barrier_reduction_{mode}'
    if col in df_v2.columns:
        axes[0,1].scatter(df_v2['screening_energy_eV'], df_v2[col],
                         alpha=0.1, s=3, color=color, label=mode)
axes[0,1].set_xlabel('Screening Energy (eV)')
axes[0,1].set_ylabel('Barrier Reduction Ratio')
axes[0,1].set_title('Screening vs Barrier Reduction')
axes[0,1].legend(fontsize=8)

# 3. Work function vs excess heat
mask_heat = df_v2['has_excess_heat'] == 1
axes[0,2].scatter(df_v2.loc[~mask_heat, 'work_function_eV'],
                  df_v2.loc[~mask_heat, 'excess_heat_W'], alpha=0.1, s=3, color='gray', label='No heat')
axes[0,2].scatter(df_v2.loc[mask_heat, 'work_function_eV'],
                  df_v2.loc[mask_heat, 'excess_heat_W'], alpha=0.3, s=10, color='#ff7043', label='Heat')
axes[0,2].set_xlabel('Work Function (eV)')
axes[0,2].set_ylabel('Excess Heat (W)')
axes[0,2].set_title('Work Function vs Excess Heat')
axes[0,2].legend(fontsize=8)

# 4. Loading ratio vs phase transition
axes[1,0].scatter(df_v2['loading_ratio'], df_v2['phase_encoded'],
                  c=df_v2['excess_heat_W'], cmap='hot', alpha=0.3, s=5)
axes[1,0].set_xlabel('Loading Ratio (D/M or H/M)')
axes[1,0].set_ylabel('Phase (encoded)')
axes[1,0].set_title('Loading vs Phase Transition')
cb = plt.colorbar(axes[1,0].collections[0], ax=axes[1,0])
cb.set_label('Excess Heat (W)')

# 5. Fermi energy vs neutron detection
if 'has_neutrons' in df_v2.columns:
    mask_n = df_v2['has_neutrons'] == 1
    axes[1,1].scatter(df_v2.loc[~mask_n, 'fermi_energy_eV'],
                      df_v2.loc[~mask_n, 'temperature_K'], alpha=0.1, s=3, color='gray', label='No neutrons')
    axes[1,1].scatter(df_v2.loc[mask_n, 'fermi_energy_eV'],
                      df_v2.loc[mask_n, 'temperature_K'], alpha=0.5, s=20, color='red', label='Neutrons')
    axes[1,1].set_xlabel('Fermi Energy (eV)')
    axes[1,1].set_ylabel('Temperature (K)')
    axes[1,1].set_title('Fermi Energy vs T (neutron events)')
    axes[1,1].legend(fontsize=8)

# 6. Bulk modulus vs energy density
axes[1,2].scatter(df_v2['bulk_modulus_GPa'], df_v2['energy_density_MJ_kg'],
                  c=df_v2['has_excess_heat'], cmap='coolwarm', alpha=0.3, s=5)
axes[1,2].set_xlabel('Bulk Modulus (GPa)')
axes[1,2].set_ylabel('Energy Density (MJ/kg)')
axes[1,2].set_title('Bulk Modulus vs Energy Density')

# 7. Correlation heatmap for V2 key features
key_v2 = ['screening_energy_eV', 'loading_ratio', 'temperature_K',
          'work_function_eV', 'bulk_modulus_GPa', 'fermi_energy_eV',
          'barrier_reduction_cherepanov', 'excess_heat_W', 'COP']
key_v2_present = [c for c in key_v2 if c in df_v2.columns]
corr_v2 = df_v2[key_v2_present].corr()
sns.heatmap(corr_v2, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2,0],
            xticklabels=[c[:14] for c in key_v2_present],
            yticklabels=[c[:14] for c in key_v2_present], annot_kws={'fontsize': 7})
axes[2,0].set_title('V2 Key Feature Correlations')

# 8. Feature group variance
group_vars = {}
for group, cols in FEATURE_COLUMNS_V2.items():
    present = [c for c in cols if c in df_v2.columns]
    if present:
        group_vars[group.replace('_group', '')] = df_v2[present].var().mean()
axes[2,1].bar(group_vars.keys(), group_vars.values(), color='#66bb6a')
axes[2,1].set_title('Mean Variance by Feature Group')
axes[2,1].tick_params(axis='x', rotation=30)
axes[2,1].set_yscale('log')

# 9. COP distribution (V2)
axes[2,2].hist(df_v2['COP'].dropna(), bins=50, color='#4fc3f7', edgecolor='white')
axes[2,2].axvline(x=1.0, color='red', linestyle='--', label='COP=1 (breakeven)')
axes[2,2].set_xlabel('COP')
axes[2,2].set_ylabel('Count')
axes[2,2].set_title(f'COP Distribution (V2, mean={df_v2["COP"].mean():.2f})')
axes[2,2].legend()

plt.tight_layout()
plt.show()

### 10.1 Визуализация V2 — Multi-Process Features

In [ ]:
# V2: Обзор фич по группам
from lenr_comprehensive_data import FEATURE_COLUMNS_V2

print('Feature groups:')
for group, cols in FEATURE_COLUMNS_V2.items():
    present = sum(1 for c in cols if c in df_v2.columns)
    print(f'  {group}: {present}/{len(cols)} columns')
    for c in cols:
        status = '✓' if c in df_v2.columns else '✗'
        print(f'    {status} {c}')

print(f'\nTotal: {len(feature_cols_v2)} features, all present: {all(c in df_v2.columns for c in feature_cols_v2)}')

In [ ]:
from data_generator_v2 import LENRDataGeneratorV2

gen_v2 = LENRDataGeneratorV2(seed=42)

# Генерация расширенного датасета
N_V2 = 10000
df_v2 = gen_v2.generate_combined(
    n_synthetic=N_V2,
    noise_level=0.05,
    include_mizuno_r19=True,
    include_mizuno_neutron=True,
    include_real_excess_heat=True,
)

feature_cols_v2 = LENRDataGeneratorV2.get_feature_columns()
cls_targets = LENRDataGeneratorV2.get_classification_targets()
reg_targets = LENRDataGeneratorV2.get_regression_targets()

print(f'V2 Dataset: {df_v2.shape}')
print(f'Features: {len(feature_cols_v2)}')
print(f'Classification targets: {cls_targets}')
print(f'Regression targets: {reg_targets}')
print(f'\nData sources:')
print(df_v2['data_source'].value_counts())
print(f'\nMaterials:')
print(df_v2['material'].value_counts())

# LENR Alternative Physics — ML Training Pipeline

**3 физических режима**: Maxwell (стандарт), Coulomb Original (1785), Cherepanov (фотонная масса)

**ML модели**:
1. XGBoost Classifier — бинарная классификация: произойдёт ли реакция?
2. DNN Regressor — предсказание избыточного тепла (W) с physics-informed loss
3. Anomaly Detector — фильтрация выбросов в данных

**Данные**: синтетические (из физического движка) + экспериментальные (Kasagi, McKubre, Fleischmann-Pons и др.)

In [ ]:
# === SETUP для Google Colab ===
# Клонируем репо и ставим зависимости
import os

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: '')())

if IN_COLAB:
    !git clone https://github.com/ORTODOX1/alternative-physik.git /content/lenr
    !pip install -q xgboost shap numba
    os.chdir('/content/lenr/python')
    print('Colab setup done. GPU:', os.environ.get('COLAB_GPU', 'CPU'))
else:
    # Локальный запуск
    os.chdir(os.path.join(os.path.dirname(os.getcwd()), ''))
    print('Local setup')

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Пути — работает и в Colab, и локально
if IN_COLAB:
    ROOT = Path('/content/lenr/python')
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT.parent))  # для lenr_constants
else:
    ROOT = Path(os.getcwd())
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT.parent))

print(f'Root: {ROOT}')
print(f'Python: {sys.version}')
print(f'NumPy: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

# Настройки отображения
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4g}'.format)

## 1. Генерация данных

In [ ]:
from data_generator import LENRDataGenerator, get_feature_columns

gen = LENRDataGenerator(seed=42)

# Синтетические данные
N_SAMPLES = 10000
df_synthetic = gen.generate_parameter_sweep(n_samples=N_SAMPLES, noise_level=0.05)
print(f'Synthetic: {df_synthetic.shape}')
print(f'Reaction rate: {df_synthetic["reaction_occurred"].mean():.1%}')
print(f'Mean excess heat (>0): {df_synthetic[df_synthetic["excess_heat_W"] > 0]["excess_heat_W"].mean():.2f} W')

# Экспериментальные данные
df_experimental = gen.generate_experimental_dataframe()
print(f'\nExperimental: {df_experimental.shape}')
print(df_experimental[['lab', 'material', 'excess_W', 'COP']].to_string())

### 1.1 Данные Мизуно R19 (55 реальных измерений)

**Источник:** ICCF-22 "Increased Excess Heat from Palladium Deposited on Nickel"
- Реактор: Ni-mesh 54g (180 mesh) + Pd coating 45-60mg
- Газ: D₂, давление 2-6320 Pa
- 55 тестов, 111 дней (Feb-May 2019)
- **Ключевое:** excess heat воспроизводим при COP 1.2-1.5, управляется температурой

In [ ]:
# Данные Мизуно R19 — 53 точки с ненулевым input (+ 2 heat-after-death)
df_mizuno = gen.generate_mizuno_dataframe()
print(f'Mizuno R19: {df_mizuno.shape}')
print(f'Excess heat range: {df_mizuno["excess_heat_W"].min():.1f} - {df_mizuno["excess_heat_W"].max():.1f} W')
print(f'Temperature range: {df_mizuno["temperature_K"].min()-273:.0f} - {df_mizuno["temperature_K"].max()-273:.0f} °C')
print(f'COP range: {df_mizuno["COP"].min():.2f} - {df_mizuno["COP"].max():.2f}')
print(f'D/Ni loading: {df_mizuno["deuterium_loading"].min():.6f} - {df_mizuno["deuterium_loading"].max():.6f}')
print(f'\nВнимание: D/Ni ratio у Мизуно ~ 0.001-0.017 (очень низкий!)')
print(f'Это парадокс: McKubre threshold = D/Pd > 0.84, но Мизуно работает при D/Ni << 0.01')

# Визуализация Мизуно
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mizuno R19: 53 реальных измерения', fontsize=14)

# T vs Excess Heat — ключевая зависимость
axes[0,0].scatter(df_mizuno['temperature_K'] - 273.15, df_mizuno['excess_heat_W'],
                  c=df_mizuno['input_power_W'], cmap='plasma', s=40, edgecolors='white', linewidth=0.5)
axes[0,0].set_xlabel('Temperature (°C)')
axes[0,0].set_ylabel('Excess Heat (W)')
axes[0,0].set_title('Температура vs Excess Heat (цвет = Input Power)')
cb = plt.colorbar(axes[0,0].collections[0], ax=axes[0,0])
cb.set_label('Input (W)')

# Empirical fit: Excess = A * exp(B * T)
from lenr_constants import MIZUNO_EMPIRICAL
T_fit = np.linspace(400, 660, 100)
A = MIZUNO_EMPIRICAL['temp_dependence']['A']
B = MIZUNO_EMPIRICAL['temp_dependence']['B_per_K']
Excess_fit = A * np.exp(B * T_fit) * 54  # 54g Ni
axes[0,0].plot(T_fit - 273.15, Excess_fit, 'r--', linewidth=2,
               label=f'Fit: {A:.1e}·exp({B}·T)·54g, R²={MIZUNO_EMPIRICAL["temp_dependence"]["R2"]}')
axes[0,0].legend(fontsize=8)

# Pressure vs Excess — independence test
axes[0,1].scatter(df_mizuno['pressure_Pa'], df_mizuno['excess_heat_W'],
                  c=df_mizuno['temperature_K'] - 273.15, cmap='coolwarm', s=40)
axes[0,1].set_xlabel('Pressure (Pa)')
axes[0,1].set_ylabel('Excess Heat (W)')
axes[0,1].set_title('Давление vs Excess (цвет = Temp °C)')
axes[0,1].set_xscale('log')
cb2 = plt.colorbar(axes[0,1].collections[0], ax=axes[0,1])
cb2.set_label('Temp (°C)')

# COP distribution
axes[1,0].hist(df_mizuno['COP'].dropna(), bins=20, color='#66bb6a', edgecolor='white')
axes[1,0].set_xlabel('COP (Output/Input)')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title(f'COP: mean={df_mizuno["COP"].mean():.2f}, max={df_mizuno["COP"].max():.2f}')
axes[1,0].axvline(x=1.0, color='red', linestyle='--', label='COP=1 (breakeven)')
axes[1,0].legend()

# D/Ni loading vs Heat per gram
axes[1,1].scatter(df_mizuno['deuterium_loading'], df_mizuno['heat_per_g_Ni'],
                  c=df_mizuno['temperature_K'] - 273.15, cmap='coolwarm', s=40)
axes[1,1].set_xlabel('D/Ni Loading Ratio')
axes[1,1].set_ylabel('Heat per gram Ni (W/g)')
axes[1,1].set_title('Loading vs Heat/g (инвертированная корреляция!)')
cb3 = plt.colorbar(axes[1,1].collections[0], ax=axes[1,1])
cb3.set_label('Temp (°C)')

plt.tight_layout()
plt.show()

print(f'\nTotal experimental data: {len(df_experimental) + len(df_mizuno)} points')
print(f'  Other labs: {len(df_experimental)}')
print(f'  Mizuno R19: {len(df_mizuno)}')

### 1.2 Нейтронные эксперименты Мизуно (SUS304 + H₂, 2020-2025)

**Источник:** "Neutrons Produced by Heating Processed Metals"
European J. Applied Physics, 2025 — [DOI](https://eu-opensci.org/index.php/ejphysics/article/view/11383)
- Реактор: SUS304 нержавейка (10×40 см, 12.7 кг) + H₂ газ (НЕ дейтерий!)
- Нейтроны на 0.7 МэВ (НЕ 2.45 МэВ от классического D-D)
- Порог: ~440°C, скорость нейтронов коррелирует с dT/dt
- **Ключевое:** нейтроны из H₂+металл → поддержка теории Видома-Ларсена (ULMN)

### 1.3 Heat-After-Death (1991): 85 МДж из 100г Pd

**Источник:** Mizuno, "Nuclear Transmutation: The Reality of Cold Fusion" (transl. Jed Rothwell)
- 37.5 литров воды испарилось за 15 дней (2268 Дж/г × 37500г = 85 МДж)
- Энергоплотность 850 кДж/г — в **27 раз** больше бензина
- Эффект стазиса: ячейка поддерживала постоянную температуру после отключения нагревателя

In [ ]:
# Нейтронные эксперименты Мизуно (SUS304 + H₂) — 10 точек
df_neutron = gen.generate_neutron_dataframe()
print(f'Mizuno Neutron (SUS304): {df_neutron.shape}')
print(f'Temperature range: {df_neutron["temperature_K"].min()-273:.0f} - {df_neutron["temperature_K"].max()-273:.0f} °C')
print(f'Input power range: {df_neutron["input_power_W"].min():.0f} - {df_neutron["input_power_W"].max():.0f} W')
print(f'Neutron rate: {df_neutron["neutron_cpm"].min():.3f} - {df_neutron["neutron_cpm"].max():.3f} c/min')

from lenr_constants import MIZUNO_HEAT_AFTER_DEATH

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mizuno: Нейтроны (SUS304+H₂) и Heat-After-Death (Pd)', fontsize=14)

# Temperature vs Neutron rate
axes[0].scatter(df_neutron['temperature_K'] - 273.15, df_neutron['neutron_cpm'],
                c='#ff7043', s=80, edgecolors='white', linewidth=0.5, zorder=5)
axes[0].axvline(x=440, color='yellow', linestyle='--', alpha=0.7, label='Threshold 440°C')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Neutron rate (counts/min)')
axes[0].set_title('T vs Neutron Rate (0.7 MeV, НЕ 2.45 МэВ!)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Input power vs Neutron rate
axes[1].scatter(df_neutron['input_power_W'], df_neutron['neutron_cpm'],
                c=df_neutron['temperature_K'] - 273.15, cmap='hot', s=80, edgecolors='white')
axes[1].set_xlabel('Input Power (W)')
axes[1].set_ylabel('Neutron rate (counts/min)')
axes[1].set_title('Input Power vs Neutron Rate')
cb = plt.colorbar(axes[1].collections[0], ax=axes[1])
cb.set_label('Temp (°C)')

# Heat-After-Death timeline
had = MIZUNO_HEAT_AFTER_DEATH
days = [0, 3, 4, 5, 8, 9, 10, 15]
water_cumulative = [0, 0, 10, 10, 20, 25, 30, 37.5]
axes[2].fill_between(days, water_cumulative, alpha=0.3, color='#4fc3f7')
axes[2].plot(days, water_cumulative, 'o-', color='#4fc3f7', linewidth=2, markersize=8)
axes[2].set_xlabel('Days after electrolysis stopped')
axes[2].set_ylabel('Cumulative water evaporated (L)')
axes[2].set_title(f'Heat-After-Death 1991: {had["total_energy_MJ"]} МДж из {had["Pd_mass_g"]}г Pd')
axes[2].annotate(f'{had["energy_density_kJg"]} кДж/г\n(27x бензин)',
                 xy=(10, 30), fontsize=10, color='#ff7043',
                 bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nВсего реальных данных:')
print(f'  Лаборатории: {len(df_experimental)}')
print(f'  Мизуно R19 (D₂+NiPd): {len(df_mizuno)}')
print(f'  Мизуно нейтроны (H₂+SUS304): {len(df_neutron)}')
print(f'  ИТОГО: {len(df_experimental) + len(df_mizuno) + len(df_neutron)} реальных точек')

In [ ]:
# Обзор данных
feature_cols = get_feature_columns()
print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'\nСтатистика:')
df_synthetic[feature_cols].describe().T

## 2. Визуализация данных

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Распределение ключевых параметров', fontsize=14)

# Энергия пучка vs вероятность реакции
axes[0,0].scatter(df_synthetic['beam_energy_keV'], df_synthetic['reaction_probability'],
                  c=df_synthetic['reaction_occurred'], cmap='coolwarm', alpha=0.3, s=5)
axes[0,0].set_xlabel('E_cm (keV)')
axes[0,0].set_ylabel('P(reaction)')
axes[0,0].set_title('Энергия vs Вероятность')

# Загрузка D/Pd vs excess heat
axes[0,1].scatter(df_synthetic['deuterium_loading'], df_synthetic['excess_heat_W'],
                  c=df_synthetic['screening_energy_eV'], cmap='viridis', alpha=0.3, s=5)
axes[0,1].set_xlabel('D/Pd загрузка')
axes[0,1].set_ylabel('Excess Heat (W)')
axes[0,1].set_title('Загрузка vs Тепло')
axes[0,1].axvline(x=0.84, color='red', linestyle='--', alpha=0.7, label='McKubre threshold')
axes[0,1].legend(fontsize=8)

# Барьерные снижения: 3 режима
for i, mode in enumerate(['maxwell', 'coulomb', 'cherepanov']):
    col = f'barrier_reduction_{mode}'
    axes[0,2].hist(df_synthetic[col], bins=50, alpha=0.5, label=mode)
axes[0,2].set_xlabel('Barrier Reduction Ratio')
axes[0,2].set_title('Снижение барьера по режимам')
axes[0,2].legend(fontsize=8)

# log rate по режимам
for mode in ['maxwell', 'coulomb', 'cherepanov']:
    col = f'log_rate_{mode}'
    axes[1,0].hist(df_synthetic[col], bins=50, alpha=0.5, label=mode)
axes[1,0].set_xlabel('log10(rate)')
axes[1,0].set_title('Распределение log(rate)')
axes[1,0].legend(fontsize=8)

# Материалы vs реакция
mat_react = df_synthetic.groupby('material')['reaction_occurred'].mean().sort_values(ascending=False)
mat_react.plot.bar(ax=axes[1,1], color='#4fc3f7')
axes[1,1].set_title('Доля реакций по материалам')
axes[1,1].set_ylabel('Fraction')

# Корреляционная матрица
key_cols = ['screening_energy_eV', 'beam_energy_keV', 'temperature_K',
            'deuterium_loading', 'reaction_probability', 'excess_heat_W']
corr = df_synthetic[key_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,2],
            xticklabels=[c[:12] for c in key_cols], yticklabels=[c[:12] for c in key_cols])
axes[1,2].set_title('Корреляции')

plt.tight_layout()
plt.show()

## 3. Детекция аномалий

In [ ]:
from models.anomaly_detector import LENRAnomalyDetector

detector = LENRAnomalyDetector(contamination=0.05)
anomaly_result = detector.fit_detect(df_synthetic, feature_cols)

print(f'Всего: {anomaly_result.n_total}')
print(f'Аномалий: {anomaly_result.n_anomalies} ({anomaly_result.anomaly_fraction:.1%})')
print(f'\nТоп фичи с аномалиями:')
for feat, count in list(anomaly_result.feature_anomaly_counts.items())[:10]:
    print(f'  {feat}: {count}')

# Физическая валидация
df_checked = detector.physics_sanity_check(df_synthetic)
print(f'\nФизически валидных: {df_checked["physics_valid"].sum()} / {len(df_checked)}')

In [ ]:
# Отфильтруем аномалии
df_clean = detector.filter_data(df_synthetic)
print(f'После фильтрации: {len(df_clean)} (было {len(df_synthetic)})')

## 4. XGBoost Classifier — Предсказание реакции

In [ ]:
from models.xgboost_classifier import LENRClassifier

classifier = LENRClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
)

clf_result = classifier.train(
    df_clean,
    feature_cols=feature_cols,
    target_col='reaction_occurred',
    test_size=0.2,
    do_cv=True,
)

print('=== XGBoost Classifier Results ===')
print(f'Accuracy:  {clf_result.accuracy:.4f}')
print(f'Precision: {clf_result.precision:.4f}')
print(f'Recall:    {clf_result.recall:.4f}')
print(f'F1:        {clf_result.f1:.4f}')
print(f'AUC-ROC:   {clf_result.auc_roc:.4f}')

if clf_result.cv_scores is not None:
    print(f'\nCV AUC-ROC: {clf_result.cv_scores.mean():.4f} +/- {clf_result.cv_scores.std():.4f}')

print(f'\nConfusion Matrix:\n{clf_result.confusion_matrix}')
print(f'\n{clf_result.classification_report}')

In [ ]:
# Feature importance
fig, ax = plt.subplots(figsize=(10, 8))
feat_imp = clf_result.feature_importance
y_pos = range(len(feat_imp))
ax.barh(y_pos, list(feat_imp.values()), color='#4fc3f7')
ax.set_yticks(y_pos)
ax.set_yticklabels(list(feat_imp.keys()), fontsize=9)
ax.set_xlabel('Importance (Gain)')
ax.set_title('XGBoost Feature Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis
try:
    import shap

    if clf_result.shap_values is not None:
        # Подготовим данные для SHAP plot
        from sklearn.model_selection import train_test_split
        X = df_clean[feature_cols].values.astype(np.float32)
        y = df_clean['reaction_occurred'].values
        X_scaled = classifier.scaler.transform(X)
        _, X_test, _, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

        fig, axes = plt.subplots(1, 2, figsize=(18, 8))

        plt.sca(axes[0])
        shap.summary_plot(clf_result.shap_values, X_test,
                         feature_names=feature_cols, show=False, plot_size=None)
        axes[0].set_title('SHAP Summary Plot')

        plt.sca(axes[1])
        shap.summary_plot(clf_result.shap_values, X_test,
                         feature_names=feature_cols, plot_type='bar', show=False, plot_size=None)
        axes[1].set_title('SHAP Bar Plot')

        plt.tight_layout()
        plt.show()
    else:
        print('SHAP values not available')
except ImportError:
    print('shap not installed: pip install shap')

## 5. DNN Regressor — Предсказание избыточного тепла

In [ ]:
from models.dnn_regressor import LENRRegressor

regressor = LENRRegressor(
    hidden_dims=[128, 64, 32],
    lr=1e-3,
    batch_size=64,
    n_epochs=200,
    patience=20,
)

reg_result = regressor.train(
    df_clean,
    feature_cols=feature_cols,
    target_col='excess_heat_W',
    test_size=0.2,
    use_physics_loss=True,
)

print('=== DNN Regressor Results ===')
print(f'MSE:  {reg_result.mse:.4f}')
print(f'RMSE: {reg_result.rmse:.4f}')
print(f'MAE:  {reg_result.mae:.4f}')
print(f'R²:   {reg_result.r2:.4f}')
print(f'Epochs: {len(reg_result.train_losses)}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(reg_result.train_losses, label='Train', alpha=0.8)
axes[0].plot(reg_result.val_losses, label='Validation', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Curves')
axes[0].legend()
axes[0].set_yscale('log')

# Feature importance (gradient-based)
feat_imp_reg = reg_result.feature_importance
y_pos = range(len(feat_imp_reg))
axes[1].barh(y_pos, list(feat_imp_reg.values()), color='#ff7043')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(list(feat_imp_reg.keys()), fontsize=9)
axes[1].set_xlabel('Gradient Importance')
axes[1].set_title('DNN Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Pred vs Actual
from sklearn.model_selection import train_test_split

X = df_clean[feature_cols].values.astype(np.float32)
y = df_clean['excess_heat_W'].values
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

df_test_pred = pd.DataFrame(X_test, columns=feature_cols)
y_pred = regressor.predict(df_test_pred)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='#4fc3f7')
max_val = max(y_test.max(), y_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.7, label='Ideal')
ax.set_xlabel('Actual Excess Heat (W)')
ax.set_ylabel('Predicted Excess Heat (W)')
ax.set_title(f'Pred vs Actual (R² = {reg_result.r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Сравнение 3 физических режимов

In [ ]:
from physics_engine import compare_modes, PhysicsEngine

# Сравнение при фиксированных условиях
materials = ['Pd', 'Ni', 'Fe', 'Ti', 'Au', 'Pt', 'PdO']
modes = ['maxwell', 'coulomb_original', 'cherepanov']

comparison_data = []
for mat in materials:
    results = compare_modes(material=mat, E_cm_keV=2.5, T_K=340, D_loading=0.9)
    for mode, r in results.items():
        comparison_data.append({
            'material': mat,
            'mode': mode,
            'barrier_keV': r.barrier_keV,
            'effective_barrier_keV': r.effective_barrier_keV,
            'penetration': r.penetration_probability,
            'rate': r.reaction_rate_relative,
            'log_rate': np.log10(max(r.reaction_rate_relative, 1e-300)),
        })

df_comp = pd.DataFrame(comparison_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Effective barrier
pivot_barrier = df_comp.pivot(index='material', columns='mode', values='effective_barrier_keV')
pivot_barrier.plot.bar(ax=axes[0], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[0].set_title('Effective Barrier (keV)')
axes[0].set_ylabel('keV')
axes[0].legend(fontsize=8)

# Log rate
pivot_rate = df_comp.pivot(index='material', columns='mode', values='log_rate')
pivot_rate.plot.bar(ax=axes[1], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[1].set_title('log10(Reaction Rate)')
axes[1].set_ylabel('log10(rate)')
axes[1].legend(fontsize=8)

# Penetration probability
pivot_pen = df_comp.pivot(index='material', columns='mode', values='penetration')
pivot_pen.plot.bar(ax=axes[2], color=['#4fc3f7', '#ff7043', '#66bb6a'])
axes[2].set_title('Penetration Probability')
axes[2].set_ylabel('P')
axes[2].set_yscale('log')
axes[2].legend(fontsize=8)

plt.suptitle('Сравнение 3 режимов: E=2.5 keV, T=340K, D/Pd=0.9', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Energy sweep: rate vs energy для Pd в 3 режимах
energies = np.linspace(0.5, 30, 100)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'maxwell': '#4fc3f7', 'coulomb_original': '#ff7043', 'cherepanov': '#66bb6a'}

for mode in modes:
    engine = PhysicsEngine(mode)
    rates = []
    for E in energies:
        r = engine.calculate_barrier('Pd', E, T_K=340, D_loading=0.9)
        rates.append(np.log10(max(r.reaction_rate_relative, 1e-300)))
    ax.plot(energies, rates, label=mode, color=colors[mode], linewidth=2)

ax.set_xlabel('E_cm (keV)')
ax.set_ylabel('log10(Reaction Rate)')
ax.set_title('Pd: Rate vs Energy (D/Pd=0.9, T=340K)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Numba Benchmark

In [ ]:
import time

try:
    from numba_kernels import (
        batch_gamow_penetration, batch_cross_section_DD,
        batch_barrier_maxwell, HAS_NUMBA,
    )
    from lenr_constants import gamow_penetration, cross_section_DD

    N = 100_000
    E_array = np.random.uniform(0.1, 50.0, N).astype(np.float64)
    Us_array = np.random.uniform(50, 600, N).astype(np.float64)
    D_array = np.random.uniform(0.0, 1.0, N).astype(np.float64)

    # Warm up Numba JIT
    _ = batch_gamow_penetration(E_array[:10])

    # Python loop
    t0 = time.perf_counter()
    python_result = np.array([gamow_penetration(e) for e in E_array])
    t_python = time.perf_counter() - t0

    # Numba batch
    t0 = time.perf_counter()
    numba_result = batch_gamow_penetration(E_array)
    t_numba = time.perf_counter() - t0

    speedup = t_python / t_numba
    print(f'Numba available: {HAS_NUMBA}')
    print(f'N = {N:,}')
    print(f'Python loop: {t_python:.3f}s')
    print(f'Numba batch: {t_numba:.3f}s')
    print(f'Speedup: {speedup:.1f}x')
    print(f'Max difference: {np.max(np.abs(python_result - numba_result)):.2e}')

    # Full barrier benchmark
    t0 = time.perf_counter()
    eb, pen, rate = batch_barrier_maxwell(E_array, Us_array, D_array)
    t_barrier = time.perf_counter() - t0
    print(f'\nBatch barrier Maxwell ({N:,} samples): {t_barrier:.3f}s')

except ImportError as e:
    print(f'Numba не установлен: {e}')
    print('pip install numba')

## 8. Сохранение моделей

In [ ]:
import os

save_dir = ROOT / 'saved_models'
os.makedirs(save_dir, exist_ok=True)

# Сохраняем классификатор
classifier.save(str(save_dir / 'xgboost_classifier.joblib'))
print(f'Classifier saved to {save_dir / "xgboost_classifier.joblib"}')

# Сохраняем регрессор
regressor.save(str(save_dir / 'dnn_regressor.pt'))
print(f'Regressor saved to {save_dir / "dnn_regressor.pt"}')

# Сохраняем детектор аномалий
detector.save(str(save_dir / 'anomaly_detector.joblib'))
print(f'Anomaly detector saved to {save_dir / "anomaly_detector.joblib"}')

# Сохраняем данные
data_dir = ROOT / 'saved_data'
os.makedirs(data_dir, exist_ok=True)
df_synthetic.to_csv(str(data_dir / 'synthetic_data.csv'), index=False)
df_clean.to_csv(str(data_dir / 'clean_data.csv'), index=False)
df_mizuno.to_csv(str(data_dir / 'mizuno_r19_data.csv'), index=False)
df_neutron.to_csv(str(data_dir / 'neutron_sus304_data.csv'), index=False)
print(f'\nData saved to {data_dir}')
print(f'  synthetic_data.csv: {len(df_synthetic)} rows')
print(f'  clean_data.csv: {len(df_clean)} rows')
print(f'  mizuno_r19_data.csv: {len(df_mizuno)} rows')
print(f'  neutron_sus304_data.csv: {len(df_neutron)} rows')

In [ ]:
# Загрузка моделей (проверка)
clf_loaded = LENRClassifier.load(str(save_dir / 'xgboost_classifier.joblib'))
reg_loaded = LENRRegressor.load(str(save_dir / 'dnn_regressor.pt'))

# Предсказание на тестовой выборке
test_sample = df_clean.sample(5, random_state=123)
print('Classifier predictions:')
probs = clf_loaded.predict(test_sample)
for i, (_, row) in enumerate(test_sample.iterrows()):
    print(f"  {row['material']} | E={row['beam_energy_keV']:.1f}keV | D/Pd={row['deuterium_loading']:.2f} | P(reaction)={probs[i]:.3f}")

print('\nRegressor predictions:')
heats = reg_loaded.predict(test_sample)
for i, (_, row) in enumerate(test_sample.iterrows()):
    print(f"  {row['material']} | actual={row['excess_heat_W']:.2f}W | predicted={heats[i]:.2f}W")

## 9. Итоговая сводка

In [ ]:
print('=' * 60)
print('LENR Alternative Physics — Training Summary')
print('=' * 60)
print(f'\nДанные:')
print(f'  Синтетических: {len(df_synthetic)}')
print(f'  После фильтрации: {len(df_clean)}')
print(f'  Экспериментальных (лаборатории): {len(df_experimental)}')
print(f'  Мизуно R19 (реальные): {len(df_mizuno)}')
print(f'  Нейтронные SUS304+H₂: {len(df_neutron)}')
print(f'  Всего реальных данных: {len(df_experimental) + len(df_mizuno) + len(df_neutron)}')
print(f'  Features: {len(feature_cols)}')
print(f'\nXGBoost Classifier:')
print(f'  AUC-ROC: {clf_result.auc_roc:.4f}')
print(f'  F1: {clf_result.f1:.4f}')
if clf_result.cv_scores is not None:
    print(f'  CV AUC: {clf_result.cv_scores.mean():.4f} ± {clf_result.cv_scores.std():.4f}')
print(f'  Top features: {", ".join(list(clf_result.feature_importance.keys())[:5])}')
print(f'\nDNN Regressor:')
print(f'  R²: {reg_result.r2:.4f}')
print(f'  RMSE: {reg_result.rmse:.4f}')
print(f'  Epochs: {len(reg_result.train_losses)}')
print(f'  Top features: {", ".join(list(reg_result.feature_importance.keys())[:5])}')
print(f'\nAnomalies: {anomaly_result.n_anomalies} ({anomaly_result.anomaly_fraction:.1%})')
print(f'\nМизуно R19 (Ni+D₂):')
print(f'  Excess Heat: {df_mizuno["excess_heat_W"].min():.1f} - {df_mizuno["excess_heat_W"].max():.1f} W')
print(f'  COP: {df_mizuno["COP"].mean():.2f} (mean), {df_mizuno["COP"].max():.2f} (max)')
print(f'  D/Ni loading: {df_mizuno["deuterium_loading"].max():.4f} (очень низкий!)')
print(f'\nМизуно SUS304+H₂ (нейтроны):')
print(f'  Экспериментов: {len(df_neutron)}')
print(f'  Температура: {df_neutron["temp_C"].min():.0f}-{df_neutron["temp_C"].max():.0f}°C')
print(f'  Нейтроны: {df_neutron["neutron_cpm"].min():.3f}-{df_neutron["neutron_cpm"].max():.3f} cpm')
print(f'  Пик энергии: 0.7 MeV (НЕ 2.45 MeV DD-синтез!)')
print(f'  Порог: ~440°C, скорость коррелирует с dT/dt')
print(f'\nHeat-After-Death (1991):')
print(f'  Испарено воды: 37.5 л (85 MJ)')
print(f'  Полная энергия: 114 MJ из 100г Pd (27x бензин)')
print(f'  Трансмутации: Pd → Cu, Zn, Fe, Cr, Ca')
print(f'\nФизические режимы:')
print(f'  Maxwell — стандартный кулоновский барьер + экранирование')
print(f'  Coulomb Original — масса электричества, плотностное взаимодействие')
print(f'  Cherepanov — фотонная масса, нет заряда, магнитный поток')

## 10. V2 — Multi-Process Dataset (64 features, 10 targets)

Расширенный генератор данных, покрывающий **ВСЕ** физические процессы в LENR:

**8 групп фич (64 штуки):**
- Material (15): Z, A, структура, решётка, Дебай, плотность, модуль, работа выхода...
- Loading (10): изотоп, загрузка, фаза, энтальпия, давление, диффузия...
- Barrier (12): screening × 3 режима, penetration × 3, rate × 3, enhancement × 3
- Thermal (8): T, P, beam energy, input power, dT/dt, диффузия, T/T_melt...
- Electrochem (5): ток, напряжение, электролит, overpotential, resistance
- Surface (6): обработка, размер частиц, слои, расширение решётки, вакансии, нано
- Stimulation (4): лазер, RF, B-field, ультразвук
- Time (4): длительность, инкубация, COP, источник данных

**10 targets (multi-task):**
- 6 classification: reaction, excess heat, neutrons, tritium, He-4, transmutation
- 4 regression: excess heat (W), COP, neutron rate (n/s), energy density (MJ/kg)